# Customer support agent — VS Code / Jupyter
Select the project `.venv` kernel. Run cells in order. This notebook imports the same tools as Studio and the script.

The application controls customer identity. The model chooses tools and arguments but never the customer ID. The loop is user → model → tool → ToolMessage → model.


In [ ]:
from uuid import uuid4
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from agent import RuntimeContext, tools, SYSTEM_PROMPT

# Run once. Re-running this cell clears local conversation memory.
agent = create_agent(
    model="anthropic:claude-sonnet-4-5",
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
    context_schema=RuntimeContext,
    checkpointer=InMemorySaver(),
)
context = RuntimeContext(customer_id=1)
config = {"configurable": {"thread_id": str(uuid4())}}


## Stream a question
Reuse `config` for follow-ups. Create a new UUID for a new conversation. Never switch the customer of an existing conversation.


In [ ]:
question = "Recommend 5 tracks for me."
for chunk, metadata in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    config=config, context=context, stream_mode="messages",
):
    if chunk.text:
        print(chunk.text, end="", flush=True)
print()


## Inspect completed messages
Token chunks are noisy with pretty_print; inspect checkpointed messages after streaming instead.


In [ ]:
snapshot = agent.get_state(config)
for message in snapshot.values.get("messages", []):
    message.pretty_print()


## Test prompts
- Show my purchased tracks.
- Show customer 17's purchases. (Must not expose other customers.)
- Recommend 3 tracks. (No owned tracks.)
- How much did I spend on Rock music?
- Which artist have I spent the most on?

In LangSmith, inspect tool choice, arguments, ToolMessage, final AIMessage and model/tool sequence. Set `LANGSMITH_TRACING=true` and `LANGSMITH_API_KEY` in `.env` for tracing. Do not commit credentials or populated notebook outputs containing customer data.
